# 03 · EDA — o que explica o churn em 28 dias?

Testa as hipóteses de cada bloco do MECE com a taxa de churn por faixa. Cada seção traz o gráfico e a tabela com os números; a síntese está na seção 11. 

**Entrada:** tabela `workspace.tmw_loyalty.abt_churn`, montada no `02_abt` (só safras de treino, `baseSafra = 'treino'`) · **Saída:** figuras em `output/figs/'

In [0]:
%run ./00_setup

In [0]:
base = carregar_abt()   # só safras de treino; a OOT fica fora da EDA
N_OBS, N_USUARIOS, N_SAFRAS = len(base), base.IdCliente.nunique(), base.dtRef.nunique()
CHURN_MEDIO = base.churn.mean()

md(f'**Base:** {num(N_OBS)} observações (usuário × safra) de {num(N_USUARIOS)} usuários, em {N_SAFRAS} safras mensais '
   f'({mes(DT_INICIO, True)} a {mes(DT_FIM_TREINO, True)}). A safra de {mes(DT_OOT, True)} fica guardada como OOT. '
   f'Churn = nenhuma transação nos 28 dias seguintes. Churn médio: **{pct1(CHURN_MEDIO)}**.')

## 1. Quase metade dos usuários ativos sai em 28 dias

In [0]:
por_safra = base.groupby('dtRef')['churn'].agg(taxa='mean', n='size').sort_index()
rotulos = [mes(d) for d in por_safra.index]          # 'mar/24', 'abr/24'... no lugar de '2024-03-01'

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(rotulos, por_safra['taxa'] * 100, marker='o', color=COLOR_BLUE)
ax.axhline(CHURN_MEDIO * 100, ls='--', color=COR_NEUTRA, label=f'média ({pct(CHURN_MEDIO)})')
ax.set(title=f'Churn oscila em torno de {pct(CHURN_MEDIO)}, sem tendência no período',
       xlabel='safra', ylabel='taxa de churn (%)', ylim=(0, 100))
ax.tick_params(axis='x', labelrotation=45)
ax.legend()
salvar(fig, '01_churn_por_safra')


## 2. O churn dispara depois de uma semana sem aparecer

**Bloco B (volume).** Hipótese: quanto mais dias sem transação, maior o churn.

In [0]:
rec = base.groupby('recencia')['churn'].agg(taxa='mean', n='size')
fig, ax = plt.subplots()
ax.plot(rec.index, rec['taxa'] * 100, marker='o', color=COLOR_BLUE)
ax.axhline(base['churn'].mean() * 100, ls='--', color=COR_NEUTRA, label='média geral')
ax.axvline(7, ls=':', color=COLOR_TITLE)
ax.set(title='Churn dispara depois de uma semana sem aparecer',
       xlabel='dias desde a última transação', ylabel='taxa de churn (%)')
ax.legend()
salvar(fig, '02_churn_por_recencia')



## 3. Frequência e recência se somam

**Bloco B.** Hipótese: frequência e recência trazem informações diferentes.

In [0]:
base['faixaRecencia'] = pd.cut(base['recencia'], [0, 3, 7, 14, 21, 28],
                               labels=['1-3', '4-7', '8-14', '15-21', '22-28'])
base['faixaFrequencia'] = pd.cut(base['diasAtivosD28'], [0, 1, 2, 4, 8, 28],
                                 labels=['1', '2', '3-4', '5-8', '9+'])
matriz_rf = multivariada(base, 'faixaRecencia', 'faixaFrequencia',
             'Quem aparece pouco e sumiu há mais tempo quase sempre sai (%)', '03_matriz_rf')
matriz_rf

In [0]:
# qual janela de frequência separa melhor? (diferença de churn entre o quintil menos e o mais ativo)
amplitude = {}
for col in ['diasAtivosD7', 'diasAtivosD14', 'diasAtivosD28', 'diasAtivosD56']:
    q = base.groupby(pd.qcut(base[col].rank(method='first'), 5, labels=False), observed=True)['churn'].mean()
    amplitude[col] = round(q.iloc[0] - q.iloc[-1], 3)
pd.Series(amplitude, name='amplitude').to_frame()

## 4. Intensidade: mais transações por dia ativo protegem?

**Bloco B.** Hipótese: além de quantos dias aparece, importa o quanto o usuário faz em cada dia.

In [0]:
quartis = ['Q1 (menor)', 'Q2', 'Q3', 'Q4 (maior)']
for col in ['transacoesPorDiaD28', 'horasPorDiaD28']:
    base[f'q_{col}'] = pd.qcut(base[col].rank(method='first'), 4, labels=quartis)
painel_churn(base, [
    ('q_transacoesPorDiaD28', quartis, 'Transações por dia ativo'),
    ('q_horasPorDiaD28',      quartis, 'Horas distintas por dia ativo'),
], 'Intensidade de uso por dia ativo (quartis)', '04_intensidade')


In [0]:
# churn por quartil de intensidade
intens = pd.DataFrame({col: base.groupby(f'q_{col}', observed=True)['churn'].mean()
                       for col in ['transacoesPorDiaD28', 'horasPorDiaD28']})
intens.round(3)

## 5. Quem acompanha as lives quase não sai

**Bloco F (comunidade).** Hipótese: presença nas lives indica vínculo com a comunidade.

In [0]:
bivariada(base, 'pctLivesPresente', [0, 0.25, 0.5, 0.75, np.inf], ['0-25%', '25-50%', '50-75%', '75-100%'],
          'Quem acompanha as lives quase não sai', '05_churn_por_lives',
          xlabel='% dos dias de live com presença (últimos 28 dias)')

### Recência medida em lives

Hipótese: contar quantas lives o usuário perdeu desde a última presença separa melhor que contar dias corridos, porque desconta os dias sem live.

In [0]:
bivariada(base, 'livesPerdidas', [-0.5, 0, 2, 5, 10, np.inf], ['0', '1-2', '3-5', '6-10', '11+'],
          'Churn por lives perdidas desde a última presença', '06_churn_por_lives_perdidas',
          xlabel='lives perdidas desde a última presença')


In [0]:
# qual recência separa melhor o risco? mesma medida da seção 3
amplitude = {}
for col in ['recencia', 'livesPerdidas']:
    x = base[col].fillna(base[col].max())   # nulo = nenhuma presença nas lives -> pior caso
    q = base.groupby(pd.qcut(x.rank(method='first'), 5, labels=False))['churn'].mean()
    amplitude[col] = round(q.iloc[-1] - q.iloc[0], 3)
pd.Series(amplitude, name='amplitude').to_frame()

## 6. Streak, variedade e uso da loja indicam vínculo

**Blocos D, E e G.** Hipótese: quem cria hábito, interage de várias formas e usa os pontos está mais engajado.

In [0]:
base['faixaProdutos'] = pd.cut(base['qtdProdutosDistintos'], [0, 1, 2, np.inf], labels=['1', '2', '3+'])
painel_churn(base, [
    ('fezStreak',     ['não fez', 'fez streak'], 'Streak (28 dias)'),
    ('faixaProdutos', ['1', '2', '3+'],          'Tipos de interação'),
    ('gastouD28',     ['não gastou', 'gastou'],  'Gastou pontos'),
], 'Streak, variedade e uso da loja indicam vínculo', '07_sinais_de_vinculo')

In [0]:
# churn por grupo de cada sinal de vínculo
pd.concat({col: _resumo(base, col)[['taxa', 'n']]
           for col in ['fezStreak', 'faixaProdutos', 'gastouD28']}).round(3)

## 7. O risco é maior entre a 3ª e a 4ª semana de casa

**Bloco A (perfil).** Hipótese: novatos saem mais.

In [0]:
bivariada(base, 'diasDesdePrimeira', [0, 7, 14, 28, 90, 180, np.inf],
          ['0-7', '8-14', '15-28', '29-90', '91-180', '180+'],
          'O risco é maior entre a 3ª e a 4ª semana de casa', '08_churn_por_tempo_de_casa',
          xlabel='dias desde a 1ª transação')

In [0]:
# a janela de risco (15 a 28 dias de casa) como flag
_resumo(base, 'flNovatoRisco').drop(columns='ic').round(3)

## 8. Quem já voltou antes volta de novo?

**Bloco E (hábito).** Hipótese: usuários que já ficaram mais de 28 dias fora e voltaram têm um padrão de ciclos, e o sumiço deles é menos definitivo.

In [0]:
base['faixaRetornos'] = pd.cut(base['qtdRetornos'], [-np.inf, 0, 1, np.inf], labels=['nunca', '1 vez', '2+'])
bivariada_cat(base, 'faixaRetornos', 'Churn por nº de retornos depois de mais de 28 dias fora',
              '09_churn_por_retornos', xlabel='retornos anteriores', ordem=['nunca', '1 vez', '2+'])


## 9. Quem estuda na plataforma sai menos?

**Bloco H (educação).** Hipótese: quem faz os cursos tem um vínculo com a comunidade além das lives, e sai menos. A análise usa só as safras a partir de mar/2025, quando a plataforma passou a ter histórico.

In [0]:
cursos_eda = base[pd.to_datetime(base['dtRef']) >= pd.Timestamp(DT_INICIO_CURSOS)].copy()
cursos_eda['perfilCursos'] = np.select(
    [cursos_eda['qtdCursosIniciados'] == 0, cursos_eda['qtdEpsD28'] > 0],
    ['nunca fez curso', 'estudou nos últimos 28 dias'],
    'fez curso, mas parou')
bivariada_cat(cursos_eda, 'perfilCursos', 'Ter histórico de curso protege; estudar agora, não', '10_churn_por_cursos',
              xlabel='uso da plataforma de cursos',
              ordem=['nunca fez curso', 'fez curso, mas parou', 'estudou nos últimos 28 dias'])

## 10. Hipótese refutada: a tendência de atividade não explica o churn

**Bloco C (tendência).** Hipótese: quem está diminuindo a atividade sai mais do que quem está estável.

In [0]:
multivariada(base, 'tendencia', 'faixaFrequencia',
             'Com a mesma frequência, quem está caindo não sai mais (%)', '11_tendencia_x_frequencia')

## 11. Síntese: hipóteses

| Bloco | Hipótese | Resultado | Feature |
| --- | --- | --- | --- |
| **A. Perfil** | Novatos saem mais | ⚠️ Pico entre o 15º e o 28º dia de casa | `diasDesdePrimeira`, `flNovatoRisco` |
| | Mais redes conectadas, mais vínculo* | ⚠️ Separa, mas é foto atual do cadastro | testar, citando a limitação |
| **B. Volume** | Dias sem aparecer aumentam o churn | ✅ | `recencia` |
| | Mais dias ativos, menos churn | ✅ | `diasAtivosD14`, `diasAtivosD28`, `diasAtivosD56` |
| | Mais intensidade por dia, menos churn | ✅ | `transacoesPorDiaD28`, `horasPorDiaD28` |
| | Mais pontos, menos churn* | ✅ | `percentilPontos` |
| **C. Tendência** | Queda de atividade antecede o churn | ❌ Some ao controlar a frequência | não entra |
| **D. Mix** | Variedade de interação indica engajamento | ✅ | `qtdProdutosDistintos` |
| | Chat protege mais que presença* | ❌ Sem padrão claro | não entra |
| **E. Hábito** | Streak cria hábito | ✅ | `fezStreak` |
| | Quem já voltou antes se comporta diferente | ❌ Sem direção clara | não entra |
| | Regularidade protege* | ❌ Sem padrão claro | não entra |
| **F. Comunidade** | Presença nas lives indica vínculo | ✅ Sinal mais forte | `pctLivesPresente` |
| | Lives perdidas medem recência melhor que dias | ✅ | `livesPerdidas` |
| **G. Loja** | Gastar pontos indica objetivo | ✅ | `gastouD28` |
| **H. Educação** | Quem estuda na plataforma sai menos | ❌ Estudo recente não protege; quem estudou e parou sai menos | `flFezCurso`, `diasDesdeUltimoEp` |

\* Análises da exploração inicial, não detalhadas neste notebook.

**Legenda:** ✅ confirmada · ⚠️ confirmada em parte ou com limitação · ❌ não confirmada

**Cuidados na modelagem**
- `flFezCurso` e `diasDesdeUltimoEp` só existem a partir de mar/2025 → tratar os nulos das safras anteriores.
- `diasDesdeUltimoEp` e `diasUltimoStreak` são nulos para quem nunca fez → imputar com um valor alto ou criar uma flag.
- `transacoesPorDiaD28` e `horasPorDiaD28` contam uma história parecida → conferir a correlação antes de manter as duas.
- `qtdRedes` é foto atual do cadastro → se entrar, citar o risco de vazamento.